巡回セールス問題(TSP)の厳密解を求める.4都市。

In [ ]:
import itertools

def solve_tsp_exact(distance_matrix):
    num_cities = len(distance_matrix)
    start_city = 0  # 出発都市を「都市0」に固定します

    # 出発都市以外の都市のリスト (1, 2, 3)
    other_cities = [i for i in range(num_cities) if i != start_city]

    min_distance = float('inf')  # 最小距離を無限大で初期化
    best_path = []

    # 出発都市以外の都市のすべての並び替え（順列）を試す
    for perm in itertools.permutations(other_cities):
        current_distance = 0
        current_city = start_city
        current_path = [start_city]

        # 順列に従って都市を訪問
        for next_city in perm:
            current_distance += distance_matrix[current_city][next_city]
            current_city = next_city
            current_path.append(current_city)

        # 最後に出発都市に戻る距離を加算
        current_distance += distance_matrix[current_city][start_city]
        current_path.append(start_city)

        # これまでに見つかったルートより短ければ更新
        if current_distance < min_distance:
            min_distance = current_distance
            best_path = current_path

    return best_path, min_distance

# --- データの準備 ---
# 4都市の距離行列（例）
# distance_matrix[i][j] は 都市i から 都市j への距離を表します
# (例: 都市0から都市1への距離は 10)
distance_matrix = [
    [0, 10, 15, 20],
    [10, 0, 35, 25],
    [15, 35, 0, 30],
    [20, 25, 30, 0]
]

# 厳密解の計算
best_route, min_dist = solve_tsp_exact(distance_matrix)

# 結果の表示
print("--- 巡回セールスマン問題 (都市数: 4) の厳密解 ---")
print(f"最短ルート: {' -> '.join(map(str, best_route))}")
print(f"最短移動距離: {min_dist}")

全ての都市の巡回とコストも出力する。

In [ ]:
import itertools

def solve_and_print_all_tsp(distance_matrix):
    num_cities = len(distance_matrix)
    start_city = 0  # 出発都市を「都市0」に固定

    other_cities = [i for i in range(num_cities) if i != start_city]

    min_distance = float('inf')
    best_path = []

    print("--- 全ての巡回ルートとコストの一覧 ---")

    # すべての順列（ルート）をループで回す
    for perm in itertools.permutations(other_cities):
        current_distance = 0
        current_city = start_city
        current_path = [start_city]

        # ルートに沿って移動
        for next_city in perm:
            current_distance += distance_matrix[current_city][next_city]
            current_city = next_city
            current_path.append(current_city)

        # 出発都市に戻る
        current_distance += distance_matrix[current_city][start_city]
        current_path.append(start_city)

        # 経路の文字列を作成して表示
        route_str = " -> ".join(map(str, current_path))
        print(f"ルート: {route_str}  [コスト: {current_distance}]")

        # 最短ルートの更新チェック
        if current_distance < min_distance:
            min_distance = current_distance
            best_path = current_path

    print("-" * 35)
    print(f"★ 最短ルート: {' -> '.join(map(str, best_path))}")
    print(f"★ 最小コスト: {min_distance}")

# --- データの準備（4都市の距離行列） ---
distance_matrix = [
    [0, 10, 15, 20],
    [10, 0, 35, 25],
    [15, 35, 0, 30],
    [20, 25, 30, 0]
]

# 実行
solve_and_print_all_tsp(distance_matrix)

best5を出力する

In [ ]:
import itertools

def solve_and_print_top5_tsp(distance_matrix):
    num_cities = len(distance_matrix)
    start_city = 0  # 出発都市を「都市0」に固定

    other_cities = [i for i in range(num_cities) if i != start_city]

    # 計算したすべての（ルート, コスト）を保存するリスト
    all_routes = []

    # すべてのルートを計算
    for perm in itertools.permutations(other_cities):
        current_distance = 0
        current_city = start_city
        current_path = [start_city]

        # ルートに沿って移動
        for next_city in perm:
            current_distance += distance_matrix[current_city][next_city]
            current_city = next_city
            current_path.append(current_city)

        # 出発都市に戻る
        current_distance += distance_matrix[current_city][start_city]
        current_path.append(start_city)

        # リストに追加
        all_routes.append((current_path, current_distance))

    # コスト（要素の2番目: x[1]）が小さい順（昇順）にソート
    all_routes.sort(key=lambda x: x[1])

    print("--- 巡回ルート コストが低い順ベスト5 ---")

    # 上位5件をループで表示（全ルートが5件未満の場合も考慮してスライスを使用）
    for rank, (path, distance) in enumerate(all_routes[:5], 1):
        route_str = " -> ".join(map(str, path))
        print(f"{rank}位: {route_str}  [コスト: {distance}]")

# --- データの準備（4都市の距離行列） ---
distance_matrix = [
    [0, 10, 15, 20],
    [10, 0, 35, 25],
    [15, 35, 0, 30],
    [20, 25, 30, 0]
]

# 実行
solve_and_print_top5_tsp(distance_matrix)

都市を10にして、近似解法で解いてみる。

In [ ]:
import math
import random
import time  # 時間計測用のライブラリを追加

def get_total_distance(tour, distance_matrix):
    """ルートの総移動距離（コスト）を計算する関数"""
    dist = 0
    n = len(tour)
    for i in range(n):
        dist += distance_matrix[tour[i]][tour[(i + 1) % n]]
    return dist

def two_opt(tour, distance_matrix):
    """2-opt法：ルートの交差をほどいて改善する局所探索"""
    n = len(tour)
    best_tour = tour[:]
    best_dist = get_total_distance(best_tour, distance_matrix)
    improved = True

    while improved:
        improved = False
        for i in range(1, n - 1):
            for j in range(i + 1, n):
                new_tour = best_tour[:]
                new_tour[i:j+1] = reversed(best_tour[i:j+1])
                new_dist = get_total_distance(new_tour, distance_matrix)

                if new_dist < best_dist:
                    best_tour = new_tour
                    best_dist = new_dist
                    improved = True
                    break
            if improved:
                break
    return best_tour, best_dist

def solve_tsp_approx(distance_matrix, num_iterations=50):
    """ランダムな初期解から2-optを繰り返し、ベスト5を抽出する"""
    num_cities = len(distance_matrix)
    start_city = 0
    other_cities = [i for i in range(num_cities) if i != start_city]

    unique_results = {}

    for _ in range(num_iterations):
        shuffled = other_cities[:]
        random.shuffle(shuffled)
        initial_tour = [start_city] + shuffled

        optimized_tour, dist = two_opt(initial_tour, distance_matrix)

        final_path = tuple(optimized_tour + [start_city])
        unique_results[final_path] = dist

    sorted_results = sorted(unique_results.items(), key=lambda x: x[1])
    return sorted_results

# --- データの準備（10都市の座標から距離行列を自動生成） ---
cities_coords = [
    (0, 0), (10, 20), (30, 10), (40, 30), (50, 0),
    (25, 40), (60, 20), (15, 5), (45, 45), (10, 35)
]

num_cities = len(cities_coords)
distance_matrix = [[0] * num_cities for _ in range(num_cities)]

for i in range(num_cities):
    for j in range(num_cities):
        x1, y1 = cities_coords[i]
        x2, y2 = cities_coords[j]
        distance_matrix[i][j] = int(math.sqrt((x1 - x2)**2 + (y1 - y2)**2))


# ==========================================
# 1. 計算開始前の時間を記録
# ==========================================
start_time = time.perf_counter()

# 近似解法の実行（50回アプローチを変えて探索）
best_5_routes = solve_tsp_approx(distance_matrix, num_iterations=50)

# ==========================================
# 2. 計算終了後の時間を記録して差分を計算
# ==========================================
end_time = time.perf_counter()
execution_time_ms = (end_time - start_time) * 1000  # 秒をミリ秒に変換


# 結果の表示
print(f"--- 巡回ルート 近似解法によるベスト5 (都市数: {num_cities}) ---")
for rank, (path, distance) in enumerate(best_5_routes[:5], 1):
    route_str = " -> ".join(map(str, path))
    print(f"{rank}位: {route_str}  [コスト: {distance}]")

print("-" * 45)
# 小数点以下2桁まで処理時間を表示
print(f"⏱ 処理時間: {execution_time_ms:.2f} ミリ秒")

10都市の厳密解

In [ ]:
import math
import itertools
import time

def solve_tsp_exact_top5(distance_matrix):
    num_cities = len(distance_matrix)
    start_city = 0  # 出発都市を「都市0」に固定
    other_cities = [i for i in range(num_cities) if i != start_city]

    all_routes = []

    # 9! = 362,880 通りのすべての順列を網羅する
    for perm in itertools.permutations(other_cities):
        current_distance = 0
        current_city = start_city
        current_path = [start_city]

        # ルートに沿って移動コストを計算
        for next_city in perm:
            current_distance += distance_matrix[current_city][next_city]
            current_city = next_city
            current_path.append(current_city)

        # 出発都市に戻る
        current_distance += distance_matrix[current_city][start_city]
        current_path.append(start_city)

        # すべての結果を記憶
        all_routes.append((current_path, current_distance))

    # コストが低い順（昇順）にソート
    all_routes.sort(key=lambda x: x[1])
    return all_routes[:5]

# --- データの準備（前回と同じ10都市の座標） ---
cities_coords = [
    (0, 0), (10, 20), (30, 10), (40, 30), (50, 0),
    (25, 40), (60, 20), (15, 5), (45, 45), (10, 35)
]

num_cities = len(cities_coords)
distance_matrix = [[0] * num_cities for _ in range(num_cities)]

for i in range(num_cities):
    for j in range(num_cities):
        x1, y1 = cities_coords[i]
        x2, y2 = cities_coords[j]
        distance_matrix[i][j] = int(math.sqrt((x1 - x2)**2 + (y1 - y2)**2))


# ==========================================
# 時間計測の開始
# ==========================================
start_time = time.perf_counter()

# 厳密解の計算（ベスト5を取得）
best_5_routes = solve_tsp_exact_top5(distance_matrix)

# ==========================================
# 時間計測の終了
# ==========================================
end_time = time.perf_counter()
execution_time_ms = (end_time - start_time) * 1000


# 結果の表示
print(f"--- 巡回ルート 厳密解法によるベスト5 (都市数: {num_cities}) ---")
for rank, (path, distance) in enumerate(best_5_routes, 1):
    route_str = " -> ".join(map(str, path))
    print(f"{rank}位: {route_str}  [コスト: {distance}]")

print("-" * 45)
print(f"⏱ 処理時間: {execution_time_ms:.2f} ミリ秒")

100都市での近似解法(最近傍法）

In [ ]:
import math
import random
import time

def get_total_distance(tour, distance_matrix):
    """ルートの総移動距離（コスト）を計算する関数"""
    dist = 0
    n = len(tour)
    for i in range(n):
        dist += distance_matrix[tour[i]][tour[(i + 1) % n]]
    return dist

def nearest_neighbor_tour(start_city, distance_matrix):
    """最近傍法（Nearest Neighbor）: 常に一番近い未訪問の都市を選ぶ"""
    n = len(distance_matrix)
    unvisited = set(range(n))
    unvisited.remove(start_city)
    tour = [start_city]
    current = start_city

    while unvisited:
        # 現在地から一番近い未訪問の都市を探す
        next_city = min(unvisited, key=lambda c: distance_matrix[current][c])
        unvisited.remove(next_city)
        tour.append(next_city)
        current = next_city
    return tour


# --- データの準備（100都市の座標をランダム生成） ---
random.seed(42)  # 毎回同じ結果になるようにシードを固定
num_cities = 100
cities_coords = [(random.randint(0, 1000), random.randint(0, 1000)) for _ in range(num_cities)]

# 距離行列の生成
distance_matrix = [[0] * num_cities for _ in range(num_cities)]
for i in range(num_cities):
    for j in range(num_cities):
        x1, y1 = cities_coords[i]
        x2, y2 = cities_coords[j]
        distance_matrix[i][j] = int(math.sqrt((x1 - x2)**2 + (y1 - y2)**2))


# ==========================================
# 時間計測の開始
# ==========================================
start_time = time.perf_counter()

unique_results = {}
# 100個すべての都市を出発点として、それぞれ最近傍法でルートを作成する
for start in range(num_cities):
    # 最近傍法によるルート構築
    tour = nearest_neighbor_tour(start, distance_matrix)

    # 構築したルートの総コストを計算
    dist = get_total_distance(tour, distance_matrix)

    # どの場合も「都市0」からスタートして0に戻る見た目に統一（ルートを回転）
    zero_idx = tour.index(0)
    rotated_tour = tour[zero_idx:] + tour[:zero_idx]
    final_path = tuple(rotated_tour + [0])

    unique_results[final_path] = dist

# コストが低い順にソート
sorted_results = sorted(unique_results.items(), key=lambda x: x[1])

# ==========================================
# 時間計測の終了
# ==========================================
end_time = time.perf_counter()
execution_time_ms = (end_time - start_time) * 1000


# 結果の表示
print(f"--- 巡回ルート 最近傍法によるベスト5 (都市数: {num_cities}) ---")
print("(※100都市は長いため、経路表示を一部省略しています)")
print("-" * 55)

for rank, (path, distance) in enumerate(sorted_results[:5], 1):
    # 最初と最後の数都市だけを表示
    path_str = f"{path[0]} -> {path[1]} -> {path[2]} ... {path[-3]} -> {path[-2]} -> {path[-1]}"
    print(f"{rank}位: {path_str}  [コスト: {distance}]")

print("-" * 55)
print(f"⏱ 処理時間: {execution_time_ms:.2f} ミリ秒")

100都市で、2-opt法の近似解法で解く。

In [ ]:
import math
import random
import time

def get_total_distance(tour, distance_matrix):
    """ルートの総移動距離（コスト）を計算する関数"""
    dist = 0
    n = len(tour)
    for i in range(n):
        dist += distance_matrix[tour[i]][tour[(i + 1) % n]]
    return dist

def two_opt(tour, distance_matrix):
    """純粋な2-opt法: 2本の枝を繋ぎ替えて交差をほどく"""
    n = len(tour)
    best_tour = tour[:]
    best_dist = get_total_distance(best_tour, distance_matrix)
    improved = True

    while improved:
        improved = False
        # すべての2本の枝の組み合わせを試す
        for i in range(1, n - 1):
            for j in range(i + 1, n):
                idx_i_minus = i - 1
                idx_j_plus = (j + 1) % n

                # 繋ぎ替える4地点の距離の差分（デルタ）を計算
                old_dist = distance_matrix[best_tour[idx_i_minus]][best_tour[i]] + distance_matrix[best_tour[j]][best_tour[idx_j_plus]]
                new_dist = distance_matrix[best_tour[idx_i_minus]][best_tour[j]] + distance_matrix[best_tour[i]][best_tour[idx_j_plus]]

                # ルートを繋ぎ替えた方が短くなる場合
                if new_dist < old_dist:
                    # iからjまでの区間を逆順にする
                    best_tour[i:j+1] = reversed(best_tour[i:j+1])
                    best_dist = best_dist - old_dist + new_dist
                    improved = True
                    break
            if improved:
                break
    return best_tour, best_dist

# --- データの準備（100都市の座標をランダム生成） ---
random.seed(42)  # 毎回同じ結果になるように固定
num_cities = 100
cities_coords = [(random.randint(0, 1000), random.randint(0, 1000)) for _ in range(num_cities)]

# 距離行列の生成
distance_matrix = [[0] * num_cities for _ in range(num_cities)]
for i in range(num_cities):
    for j in range(num_cities):
        x1, y1 = cities_coords[i]
        x2, y2 = cities_coords[j]
        distance_matrix[i][j] = int(math.sqrt((x1 - x2)**2 + (y1 - y2)**2))


# ==========================================
# 時間計測の開始
# ==========================================
start_time = time.perf_counter()

unique_results = {}
# 50回の異なる「完全ランダムな初期ルート」から2-optを実行
for _ in range(50):
    other_cities = list(range(1, num_cities))
    random.shuffle(other_cities)  # 完全にシャッフル（ランダム化）
    init_tour = [0] + other_cities

    # 2-optだけで最適化
    opt_tour, dist = two_opt(init_tour, distance_matrix)

    # 表示用に都市0スタートに統一
    zero_idx = opt_tour.index(0)
    rotated_tour = opt_tour[zero_idx:] + opt_tour[:zero_idx]
    final_path = tuple(rotated_tour + [0])

    unique_results[final_path] = dist

# コストが低い順にソート
sorted_results = sorted(unique_results.items(), key=lambda x: x[1])

# ==========================================
# 時間計測の終了
# ==========================================
end_time = time.perf_counter()
execution_time_ms = (end_time - start_time) * 1000


# 結果の表示
print(f"--- 巡回ルート 純粋な2-opt法によるベスト5 (都市数: {num_cities}) ---")
print("(※初期解をすべてランダムに生成し、2-optのみで最適化しました)")
print("-" * 55)

for rank, (path, distance) in enumerate(sorted_results[:5], 1):
    path_str = f"{path[0]} -> {path[1]} -> {path[2]} ... {path[-3]} -> {path[-2]} -> {path[-1]}"
    print(f"{rank}位: {path_str}  [コスト: {distance}]")

print("-" * 55)
print(f"⏱ 処理時間 (50回試行の合計): {execution_time_ms:.2f} ミリ秒")

#**課題のプログラムのセル**

指定処理時間内で打ち切る近似解法。時間が来たら、その時までの近似解の候補を出力する。

In [1]:
import math
import random
import time

def get_total_distance(tour, distance_matrix):
    """ルートの総移動距離（コスト）を計算する関数"""
    dist = 0
    n = len(tour)
    for i in range(n):
        dist += distance_matrix[tour[i]][tour[(i + 1) % n]]
    return dist

def two_opt(tour, distance_matrix):
    """純粋な2-opt法: 2本の枝を繋ぎ替えて交差をほどく"""
    n = len(tour)
    best_tour = tour[:]
    best_dist = get_total_distance(best_tour, distance_matrix)
    improved = True

    while improved:
        improved = False
        for i in range(1, n - 1):
            for j in range(i + 1, n):
                idx_i_minus = i - 1
                idx_j_plus = (j + 1) % n

                # 繋ぎ替える4地点の距離の差分（デルタ）を計算
                old_dist = distance_matrix[best_tour[idx_i_minus]][best_tour[i]] + distance_matrix[best_tour[j]][best_tour[idx_j_plus]]
                new_dist = distance_matrix[best_tour[idx_i_minus]][best_tour[j]] + distance_matrix[best_tour[i]][best_tour[idx_j_plus]]

                if new_dist < old_dist:
                    best_tour[i:j+1] = reversed(best_tour[i:j+1])
                    best_dist = best_dist - old_dist + new_dist
                    improved = True
                    break
            if improved:
                break
    return best_tour, best_dist

# --- データの準備（100都市の座標をランダム生成） ---
random.seed(42)  # 毎回同じ結果になるように固定
num_cities = 100
cities_coords = [(random.randint(0, 1000), random.randint(0, 1000)) for _ in range(num_cities)]

# 距離行列の生成
distance_matrix = [[0] * num_cities for _ in range(num_cities)]
for i in range(num_cities):
    for j in range(num_cities):
        x1, y1 = cities_coords[i]
        x2, y2 = cities_coords[j]
        distance_matrix[i][j] = int(math.sqrt((x1 - x2)**2 + (y1 - y2)**2))


# ==========================================
# 時間指定による探索の実行
# ==========================================
#ここを変えると処理時間を変えれる↓
LIMIT_TIME = 3.0  # 制限時間（秒）
unique_results = {}
iterations = 0    # 試行回数のカウンター

start_time = time.perf_counter()

# 開始からLIMIT_TIME秒が経過するまでループをぶん回す
while (time.perf_counter() - start_time) < LIMIT_TIME:
    iterations += 1

    # 完全ランダムな初期解を生成
    other_cities = list(range(1, num_cities))
    random.shuffle(other_cities)
    init_tour = [0] + other_cities

    # 2-optで最適化
    opt_tour, dist = two_opt(init_tour, distance_matrix)

    # 表示用に都市0スタートに一貫させる
    zero_idx = opt_tour.index(0)
    rotated_tour = opt_tour[zero_idx:] + opt_tour[:zero_idx]
    final_path = tuple(rotated_tour + [0])

    unique_results[final_path] = dist

# ループが終了した時点の最終時間を計測
end_time = time.perf_counter()
execution_time_ms = (end_time - start_time) * 1000

# コストが低い順にソート
sorted_results = sorted(unique_results.items(), key=lambda x: x[1])


# 結果の表示
print(f"--- 巡回ルート 近似解法によるベスト5 (都市数: {num_cities}) ---")
print(f"(制限時間 {LIMIT_TIME} 秒の間に全力で探索した結果です)")
print("-" * 55)

for rank, (path, distance) in enumerate(sorted_results[:5], 1):
    path_str = f"{path[0]} -> {path[1]} -> {path[2]} ... {path[-3]} -> {path[-2]} -> {path[-1]}"
    print(f"{rank}位: {path_str}  [コスト: {distance}]")

print("-" * 55)
print(f"🔄 2秒間での総試行回数: {iterations} 回")
print(f"⏱ 実際の処理時間: {execution_time_ms:.2f} ミリ秒 ({execution_time_ms/1000:.2f} 秒)")

--- 巡回ルート 近似解法によるベスト5 (都市数: 100) ---
(制限時間 3.0 秒の間に全力で探索した結果です)
-------------------------------------------------------
1位: 0 -> 89 -> 55 ... 4 -> 40 -> 0  [コスト: 7557]
2位: 0 -> 40 -> 4 ... 55 -> 89 -> 0  [コスト: 7756]
3位: 0 -> 89 -> 55 ... 4 -> 40 -> 0  [コスト: 7761]
4位: 0 -> 40 -> 4 ... 55 -> 89 -> 0  [コスト: 7810]
5位: 0 -> 40 -> 4 ... 55 -> 89 -> 0  [コスト: 7833]
-------------------------------------------------------
🔄 2秒間での総試行回数: 21 回
⏱ 実際の処理時間: 3001.05 ミリ秒 (3.00 秒)


都市数を変えて、厳密解を解く時間をグラフ化する。

線形スケールでの処理時間のグラフを出力

In [ ]:
import math
import random
import time
import matplotlib.pyplot as plt

def solve_tsp_dp(distance_matrix):
    """動的計画法によるTSP厳密解法（コストと経路の両方を返す）"""
    n = len(distance_matrix)
    memo = {}
    next_node = {}  # 経路を復元するために「次にどの都市を選んだか」を記録する辞書

    def tsp(mask, u):
        # すべての都市を訪問した場合（出発地に戻る）
        if mask == (1 << n) - 1:
            return distance_matrix[u][0]

        if (mask, u) in memo:
            return memo[(mask, u)]

        res = float('inf')
        best_next = -1

        for v in range(n):
            if not (mask & (1 << v)):
                cost = distance_matrix[u][v] + tsp(mask | (1 << v), v)
                if cost < res:
                    res = cost
                    best_next = v  # 最小コストを更新した「次の都市」を記憶

        memo[(mask, u)] = res
        next_node[(mask, u)] = best_next
        return res

    # 最小コストを計算
    min_cost = tsp(1, 0)

    # --- 経路の復元 ---
    path = [0]
    mask = 1
    u = 0
    # 残りの都市をたどる
    for _ in range(n - 1):
        v = next_node[(mask, u)]
        path.append(v)
        mask |= (1 << v)
        u = v
    path.append(0)  # 最後に出発都市に戻る

    return min_cost, path

# ==========================================================
# 実験設定
# ==========================================================
MAX_CITIES = 15

num_cities_list = list(range(5, MAX_CITIES + 1))
execution_times = []

print("--- TSP厳密解（動的計画法）の実験開始 ---")

for num_cities in num_cities_list:
    print(f"都市数 {num_cities:2d} を計算中...", end="", flush=True)

    random.seed(42)
    coords = [(random.randint(0, 1000), random.randint(0, 1000)) for _ in range(num_cities)]

    distance_matrix = [[0] * num_cities for _ in range(num_cities)]
    for i in range(num_cities):
        for j in range(num_cities):
            if i != j:
                x1, y1 = coords[i]
                x2, y2 = coords[j]
                distance_matrix[i][j] = math.sqrt((x1 - x2)**2 + (y1 - y2)**2)

    start_time = time.perf_counter()
    # コストと経路の両方を受け取る
    min_cost, best_path = solve_tsp_dp(distance_matrix)
    end_time = time.perf_counter()

    elapsed_time_ms = (end_time - start_time) * 1000
    execution_times.append(elapsed_time_ms)

    # 経路を見やすい文字列に変換
    path_str = " -> ".join(map(str, best_path))

    # ターミナルに出力（改行を入れて見やすく配置）
    print(f" 完了! ⏱ {elapsed_time_ms:7.2f} ms")
    print(f"    ┣ 🏆 コスト: {min_cost:.2f}")
    print(f"    ┗ 🗺️ 経路  : {path_str}")

print("\nすべて完了しました。グラフを描画します。")

# ==========================================================
# matplotlib による折れ線グラフの描画（リニアスケール）
# ==========================================================
plt.figure(figsize=(10, 6))
plt.plot(num_cities_list, execution_times, marker='o', color='#e377c2', linestyle='-', linewidth=2, markersize=6)

plt.title("TSP Exact Solution (DP) Execution Time - Linear Scale", fontsize=14, fontweight='bold')
plt.xlabel("Number of Cities", fontsize=12)
plt.ylabel("Execution Time (ms)", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)

plt.ylim(bottom=0)

for i, txt in enumerate(execution_times):
    plt.annotate(f"{txt:.1f}ms", (num_cities_list[i], execution_times[i]),
                 textcoords="offset points", xytext=(0,10), ha='center', fontsize=9)

plt.tight_layout()
plt.show()

2-optの近似解法で解く時間を測る

In [ ]:
import math
import random
import time
import matplotlib.pyplot as plt

def get_total_distance(tour, distance_matrix):
    """ルートの総移動距離（コスト）を計算する"""
    dist = 0
    n = len(tour)
    for i in range(n):
        dist += distance_matrix[tour[i]][tour[(i + 1) % n]]
    return dist

def solve_tsp_2opt(distance_matrix, num_iterations=50):
    """2-opt法によるTSP近似解法（コストと経路の両方を返す）"""
    n = len(distance_matrix)
    best_overall_tour = None
    best_overall_dist = float('inf')

    other_cities = list(range(1, n))

    # 指定回数（今回は50回）、異なるランダム初期解から2-optを実行する
    for _ in range(num_iterations):
        random.shuffle(other_cities)
        tour = [0] + other_cities
        current_dist = get_total_distance(tour, distance_matrix)

        improved = True
        while improved:
            improved = False
            for i in range(1, n - 1):
                for j in range(i + 1, n):
                    idx_i_minus = i - 1
                    idx_j_plus = (j + 1) % n

                    # 繋ぎ替える4地点の距離の差分だけを計算
                    old_dist = distance_matrix[tour[idx_i_minus]][tour[i]] + distance_matrix[tour[j]][tour[idx_j_plus]]
                    new_dist = distance_matrix[tour[idx_i_minus]][tour[j]] + distance_matrix[tour[i]][tour[idx_j_plus]]

                    if new_dist < old_dist:
                        # 経路の一部を逆順にして繋ぎ直す
                        tour[i:j+1] = reversed(tour[i:j+1])
                        current_dist = current_dist - old_dist + new_dist
                        improved = True
                        break
                if improved:
                    break

        # これまでに見つけた中で一番良ければ記録を更新
        if current_dist < best_overall_dist:
            best_overall_dist = current_dist
            best_overall_tour = tour[:]

    # 最後に出発都市(0)に戻るように経路を整える
    best_path = best_overall_tour + [0]
    return best_overall_dist, best_path

# ==========================================================
# 実験設定
# ==========================================================
# 近似解法なら20都市でも一瞬で終わるため、最大20都市に設定
MAX_CITIES = 20

num_cities_list = list(range(5, MAX_CITIES + 1))
execution_times = []

print("--- TSP近似解（2-opt法）の実験開始 ---")

for num_cities in num_cities_list:
    print(f"都市数 {num_cities:2d} を計算中...", end="", flush=True)

    random.seed(42)
    coords = [(random.randint(0, 1000), random.randint(0, 1000)) for _ in range(num_cities)]

    distance_matrix = [[0] * num_cities for _ in range(num_cities)]
    for i in range(num_cities):
        for j in range(num_cities):
            if i != j:
                x1, y1 = coords[i]
                x2, y2 = coords[j]
                distance_matrix[i][j] = math.sqrt((x1 - x2)**2 + (y1 - y2)**2)

    start_time = time.perf_counter()
    # 2-opt法でコストと経路を受け取る
    min_cost, best_path = solve_tsp_2opt(distance_matrix, num_iterations=50)
    end_time = time.perf_counter()

    elapsed_time_ms = (end_time - start_time) * 1000
    execution_times.append(elapsed_time_ms)

    path_str = " -> ".join(map(str, best_path))

    print(f" 完了! ⏱ {elapsed_time_ms:7.2f} ms")
    print(f"    ┣ 🏆 コスト: {min_cost:.2f}")
    print(f"    ┗ 🗺️ 経路  : {path_str}")

print("\nすべて完了しました。グラフを描画します。")

# ==========================================================
# matplotlib による折れ線グラフの描画
# ==========================================================
plt.figure(figsize=(10, 6))
plt.plot(num_cities_list, execution_times, marker='o', color='#2ca02c', linestyle='-', linewidth=2, markersize=6)

plt.title("TSP Approximate Solution (2-opt) Execution Time", fontsize=14, fontweight='bold')
plt.xlabel("Number of Cities", fontsize=12)
plt.ylabel("Execution Time (ms)", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)

plt.ylim(bottom=0)

for i, txt in enumerate(execution_times):
    plt.annotate(f"{txt:.1f}ms", (num_cities_list[i], execution_times[i]),
                 textcoords="offset points", xytext=(0,10), ha='center', fontsize=9)

plt.tight_layout()
plt.show()